# QuantJourney SDK - Portfolio Analysis

This notebook demonstrates portfolio analytics:
- Portfolio construction
- Performance tracking
- Risk metrics (Sharpe, Sortino, Max Drawdown)
- Correlation analysis
- Risk contribution by asset
- Efficient frontier concepts

**API:** https://api.quantjourney.cloud

## Run Output

![06_portfolio_analysis](../plots/06_portfolio_analysis_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from quantjourney.sdk import QuantJourneyAPI
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# API Key authentication (recommended)
import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Define Portfolio

In [ ]:
# Diversified portfolio (sector allocation)
portfolio = {
    'AAPL': {'weight': 0.15, 'sector': 'Technology'},
    'MSFT': {'weight': 0.15, 'sector': 'Technology'},
    'GOOGL': {'weight': 0.10, 'sector': 'Communication'},
    'JPM': {'weight': 0.12, 'sector': 'Financials'},
    'JNJ': {'weight': 0.12, 'sector': 'Healthcare'},
    'PG': {'weight': 0.10, 'sector': 'Consumer Staples'},
    'XOM': {'weight': 0.10, 'sector': 'Energy'},
    'HD': {'weight': 0.08, 'sector': 'Consumer Discretionary'},
    'UNP': {'weight': 0.08, 'sector': 'Industrials'}
}

symbols = list(portfolio.keys())
weights = np.array([portfolio[s]['weight'] for s in symbols])

print("Portfolio Allocation:")
for s in symbols:
    print(f"  {s}: {portfolio[s]['weight']*100:.0f}% ({portfolio[s]['sector']})")
print(f"\nTotal: {weights.sum()*100:.0f}%")


## 2. Fetch Historical Data

In [ ]:
# Fetch 2 years of data
price_data = {}

for symbol in symbols:
    response = qj.eod.get_historical_prices(
        symbol=symbol,
        start_date="2023-01-01",
        end_date="2024-12-31",
        frequency="1d"
    )
    prices = response.get('value', response) if isinstance(response, dict) else response
    if prices:
        df = pd.DataFrame(prices)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date')
        price_data[symbol] = df['close']
        print(f"✓ {symbol}: {len(df)} days")

# Combine into single DataFrame
prices_df = pd.DataFrame(price_data)
prices_df = prices_df.dropna()
print(f"\nCombined: {len(prices_df)} trading days")


## 3. Portfolio Performance

In [ ]:
# Calculate returns
returns = prices_df.pct_change().dropna()

# Portfolio returns
portfolio_returns = (returns * weights).sum(axis=1)

# Cumulative performance
cumulative = (1 + portfolio_returns).cumprod()
individual_cumulative = (1 + returns).cumprod()

# Plot
fig = go.Figure()

# Individual stocks (faded)
for col in individual_cumulative.columns:
    fig.add_trace(go.Scatter(
        x=individual_cumulative.index,
        y=(individual_cumulative[col] - 1) * 100,
        name=col,
        line=dict(width=1),
        opacity=0.4
    ))

# Portfolio (bold)
fig.add_trace(go.Scatter(
    x=cumulative.index,
    y=(cumulative - 1) * 100,
    name='PORTFOLIO',
    line=dict(width=4, color='white')
))

fig.update_layout(
    title='Portfolio vs Individual Holdings',
    xaxis_title='Date',
    yaxis_title='Return (%)',
    template='plotly_dark',
    height=550
)
fig.show()

print(f"\nTotal Portfolio Return: {(cumulative.iloc[-1]-1)*100:.1f}%")


## 4. Correlation Matrix

In [ ]:
# Calculate correlation
correlation = returns.corr()

# Heatmap
fig = px.imshow(
    correlation,
    labels=dict(color="Correlation"),
    x=correlation.columns,
    y=correlation.columns,
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1
)

fig.update_layout(
    title='Asset Correlation Matrix',
    template='plotly_dark',
    height=500,
    width=600
)
fig.show()

# Find highest and lowest correlations
corr_pairs = []
for i, s1 in enumerate(symbols):
    for j, s2 in enumerate(symbols):
        if i < j:
            corr_pairs.append((s1, s2, correlation.loc[s1, s2]))

corr_pairs.sort(key=lambda x: x[2])
print("\nLowest Correlations (best diversification):")
for s1, s2, c in corr_pairs[:3]:
    print(f"  {s1}-{s2}: {c:.2f}")

print("\nHighest Correlations:")
for s1, s2, c in corr_pairs[-3:]:
    print(f"  {s1}-{s2}: {c:.2f}")


## 5. Risk Metrics

In [ ]:
# Risk metrics calculations
trading_days = 252
risk_free_rate = 0.05  # 5% annual

# Annual return
total_return = cumulative.iloc[-1] - 1
n_years = len(portfolio_returns) / trading_days
annual_return = (1 + total_return) ** (1 / n_years) - 1

# Volatility
annual_vol = portfolio_returns.std() * np.sqrt(trading_days)

# Sharpe Ratio
sharpe = (annual_return - risk_free_rate) / annual_vol

# Sortino Ratio (downside deviation)
downside = portfolio_returns[portfolio_returns < 0]
downside_vol = downside.std() * np.sqrt(trading_days)
sortino = (annual_return - risk_free_rate) / downside_vol

# Max Drawdown
cummax = cumulative.cummax()
drawdown = (cumulative - cummax) / cummax
max_drawdown = drawdown.min()

# Calmar Ratio
calmar = annual_return / abs(max_drawdown)

# VaR (95%)
var_95 = np.percentile(portfolio_returns, 5)

# Display
metrics = {
    'Annual Return': f"{annual_return*100:.1f}%",
    'Annual Volatility': f"{annual_vol*100:.1f}%",
    'Sharpe Ratio': f"{sharpe:.2f}",
    'Sortino Ratio': f"{sortino:.2f}",
    'Max Drawdown': f"{max_drawdown*100:.1f}%",
    'Calmar Ratio': f"{calmar:.2f}",
    'VaR (95%, daily)': f"{var_95*100:.2f}%"
}

print("Portfolio Risk Metrics:")
print("=" * 35)
for k, v in metrics.items():
    print(f"  {k:20} {v:>10}")


## 6. Drawdown Analysis

In [ ]:
# Plot drawdown
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=('Portfolio Value', 'Drawdown'),
    row_heights=[0.6, 0.4],
    vertical_spacing=0.1
)

fig.add_trace(
    go.Scatter(
        x=cumulative.index, 
        y=cumulative * 100,  # $100 initial
        name='Portfolio',
        line=dict(color='cyan')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=drawdown.index, 
        y=drawdown * 100,
        fill='tozeroy',
        fillcolor='rgba(255,0,0,0.3)',
        name='Drawdown',
        line=dict(color='red')
    ),
    row=2, col=1
)

fig.update_layout(
    title='Portfolio Drawdown Analysis',
    template='plotly_dark',
    height=600
)
fig.update_yaxes(title_text='Value ($)', row=1, col=1)
fig.update_yaxes(title_text='Drawdown (%)', row=2, col=1)
fig.show()

# Worst drawdown period
max_dd_date = drawdown.idxmin()
peak_date = cumulative[:max_dd_date].idxmax()
print(f"\nMax Drawdown: {max_drawdown*100:.1f}%")
print(f"  From: {peak_date.date()}")
print(f"  To:   {max_dd_date.date()}")


## 7. Risk Contribution

In [ ]:
# Covariance matrix
cov_matrix = returns.cov() * trading_days

# Portfolio variance
port_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
port_vol = np.sqrt(port_variance)

# Marginal contribution to risk
mcr = np.dot(cov_matrix, weights) / port_vol

# Component risk (risk contribution)
risk_contribution = weights * mcr
risk_contribution_pct = risk_contribution / port_vol * 100

# Create DataFrame
risk_df = pd.DataFrame({
    'Symbol': symbols,
    'Weight (%)': weights * 100,
    'Risk Contrib (%)': risk_contribution_pct,
    'Sector': [portfolio[s]['sector'] for s in symbols]
}).sort_values('Risk Contrib (%)', ascending=False)

# Plot
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Portfolio Weights', 'Risk Contribution'),
    specs=[[{'type': 'pie'}, {'type': 'pie'}]]
)

fig.add_trace(
    go.Pie(
        labels=risk_df['Symbol'],
        values=risk_df['Weight (%)'],
        hole=0.4
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=risk_df['Symbol'],
        values=risk_df['Risk Contrib (%)'],
        hole=0.4
    ),
    row=1, col=2
)

fig.update_layout(
    title='Weight vs Risk Contribution',
    template='plotly_dark',
    height=450
)
fig.show()

print("\nRisk Contribution by Asset:")
print(risk_df.to_string(index=False))


## 8. Monthly Returns Heatmap

In [ ]:
# Monthly returns
monthly = portfolio_returns.resample('ME').apply(lambda x: (1+x).prod()-1)

# DEBUG: Check data
print("Pportfolio_returns stats:")
print(f"  Length: {len(portfolio_returns)}, Mean: {portfolio_returns.mean():.6f}, Std: {portfolio_returns.std():.6f}")
print(f"\nMonthly returns (first 6):")
print(monthly.head(6))
print(f"\nMnthly stats: min={monthly.min()*100:.2f}%, max={monthly.max()*100:.2f}%")

monthly_df = monthly.to_frame('return')
monthly_df['year'] = monthly_df.index.year
monthly_df['month'] = monthly_df.index.month

print(f"\nMonthly_df sample:")
print(monthly_df.head(6))

# Pivot for heatmap
monthly_pivot = monthly_df.pivot(index='year', columns='month', values='return') * 100

print(f"\nMonthly_pivot shape: {monthly_pivot.shape}")
print(f"Monthly_pivot columns: {list(monthly_pivot.columns)}")
print(f"Monthly_pivot:\n{monthly_pivot}")

# Rename columns to month names (only for columns that exist)
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 
               7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
monthly_pivot.columns = [month_names[m] for m in monthly_pivot.columns]

# Heatmap
fig = px.imshow(
    monthly_pivot,
    labels=dict(color="Return (%)"),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    aspect='auto'
)

# Add text annotations
for i, year in enumerate(monthly_pivot.index):
    for j, month in enumerate(monthly_pivot.columns):
        val = monthly_pivot.loc[year, month]
        if pd.notna(val):
            fig.add_annotation(
                x=j, y=i,
                text=f"{val:.1f}",
                showarrow=False,
                font=dict(size=10, color='black' if abs(val) < 3 else 'white')
            )

fig.update_layout(
    title='Monthly Returns Heatmap (%)',
    template='plotly_dark',
    height=300
)
fig.show()

# Best and worst months
print(f"\nBest Month:  {monthly.idxmax().strftime('%b %Y')} ({monthly.max()*100:.1f}%)")
print(f"Worst Month: {monthly.idxmin().strftime('%b %Y')} ({monthly.min()*100:.1f}%)")
print(f"Positive Months: {(monthly > 0).sum()}/{len(monthly)} ({(monthly > 0).mean()*100:.0f}%)")


## 9. Portfolio Summary Dashboard

In [ ]:
# Final summary
print("="*60)
print("PORTFOLIO ANALYSIS SUMMARY")
print("="*60)

print(f"\nALLOCATION ({len(symbols)} holdings)")
sector_weights = {}
for s in symbols:
    sector = portfolio[s]['sector']
    sector_weights[sector] = sector_weights.get(sector, 0) + portfolio[s]['weight']
for sector, w in sorted(sector_weights.items(), key=lambda x: -x[1]):
    print(f"   {sector}: {w*100:.0f}%")

print(f"\nERFORMANCE")
print(f"   Total Return:    {total_return*100:+.1f}%")
print(f"   Annual Return:   {annual_return*100:+.1f}%")
print(f"   Best Month:      {monthly.max()*100:+.1f}%")
print(f"   Worst Month:     {monthly.min()*100:+.1f}%")

print(f"\nRISK")
print(f"   Volatility:      {annual_vol*100:.1f}%")
print(f"   Max Drawdown:    {max_drawdown*100:.1f}%")
print(f"   VaR (95%):       {var_95*100:.2f}%")

print(f"\nRISK-ADJUSTED")
print(f"   Sharpe Ratio:    {sharpe:.2f}")
print(f"   Sortino Ratio:   {sortino:.2f}")
print(f"   Calmar Ratio:    {calmar:.2f}")

print(f"\nDIVERSIFICATION")
avg_corr = correlation.values[np.triu_indices_from(correlation.values, k=1)].mean()
print(f"   Avg Correlation: {avg_corr:.2f}")
print(f"   Effective Stocks: {1 / (weights**2).sum():.1f}")

print("\n" + "="*60)


## Summary

Portfolio analysis covered:
- **Performance**: Cumulative returns, individual vs portfolio
- **Correlation**: Heatmap, diversification benefits
- **Risk Metrics**: Sharpe, Sortino, Calmar, VaR, Max Drawdown
- **Drawdown Analysis**: Visual + worst period identification
- **Risk Contribution**: Weight vs risk allocation
- **Monthly Returns**: Seasonality heatmap